# Prepare Dataset For K-Fold CV

This notebook creates `k` dataset sets under `data/sets/set_{i}/` with splits:
- `train`
- `validate`
- `test`
- `holdout`

Each sample is saved as one compressed `.npz` file with keys:
- `image`
- `label`
- `mask`

### 1. Necessary imports
Import required libraries, set reproducibility seed, and define shared helpers.

In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import imageio.v2 as imageio
import numpy as np


def set_seed(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)

### 2. Define Configuration and Constants
Define source paths, fold count, split ratios, and output root.

In [5]:
# Editable defaults
K = 4
SEED = 42
SPLIT_RATIOS = (0.60, 0.20, 0.20)  # train, validate, test
IMAGE_EXTS = (".png",)

IMAGES_DIR = Path("data/cleaned/images")
LABELS_DIR = Path("data/cleaned/labels")
MASKS_DIR = Path("data/cleaned/masks")
OUTPUT_ROOT = Path("data/sets")

assert len(SPLIT_RATIOS) == 3, "Use exactly three ratios: train, validate, test"
assert abs(sum(SPLIT_RATIOS) - 1.0) < 1e-8, "SPLIT_RATIOS must sum to 1.0"
assert K >= 2, "K should be >= 2"

### 3. Implement Core Functions
Implement matching, fold assignments, and NPZ saving functionality.

In [6]:
@dataclass
class SampleInfo:
    stem: str
    image_path: Path
    label_path: Path | None
    mask_path: Path | None
    has_label: bool
    has_mask: bool


def list_stems(folder: Path, exts: Tuple[str, ...]) -> Dict[str, Path]:
    files = {}
    if not folder.exists():
        return files
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in exts:
            files[p.stem] = p
    return files


def read_png(path: Path) -> np.ndarray:
    return imageio.imread(path)


def save_npz_sample(
    sample_path: Path,
    image: np.ndarray,
    label: np.ndarray,
    mask: np.ndarray,
    has_label: bool,
    has_mask: bool,
    source_stem: str,
) -> None:
    sample_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        sample_path,
        image=image,
        label=label,
        mask=mask,
        has_label=np.array(has_label, dtype=np.bool_),
        has_mask=np.array(has_mask, dtype=np.bool_),
        source_stem=np.array(source_stem),
    )


def collect_samples(
    images_dir: Path,
    labels_dir: Path,
    masks_dir: Path,
    exts: Tuple[str, ...],
) -> Tuple[List[SampleInfo], List[SampleInfo]]:
    image_map = list_stems(images_dir, exts)
    label_map = list_stems(labels_dir, exts)
    mask_map = list_stems(masks_dir, exts)

    image_stems = set(image_map.keys())
    label_stems = set(label_map.keys())
    mask_stems = set(mask_map.keys())

    matched_stems = sorted(image_stems & label_stems & mask_stems)
    holdout_stems = sorted(image_stems - label_stems)

    matched = [
        SampleInfo(
            stem=s,
            image_path=image_map[s],
            label_path=label_map[s],
            mask_path=mask_map[s],
            has_label=True,
            has_mask=True,
        )
        for s in matched_stems
    ]

    holdout = [
        SampleInfo(
            stem=s,
            image_path=image_map[s],
            label_path=None,
            mask_path=mask_map.get(s),
            has_label=False,
            has_mask=(s in mask_stems),
        )
        for s in holdout_stems
    ]

    print(f"Matched count: {len(matched)}")
    print(f"Holdout count: {len(holdout)}")
    print("Matched examples:", [m.stem for m in matched[:5]])
    print("Holdout examples:", [h.stem for h in holdout[:5]])

    return matched, holdout


def create_fold_assignments(
    matched_samples: List[SampleInfo],
    k: int,
    seed: int,
    ratios: Tuple[float, float, float],
) -> Dict[int, Dict[str, List[SampleInfo]]]:
    train_r, val_r, test_r = ratios
    n = len(matched_samples)
    assignments: Dict[int, Dict[str, List[SampleInfo]]] = {}

    for fold in range(k):
        rng = np.random.default_rng(seed + fold)
        perm = rng.permutation(n)
        samples = [matched_samples[i] for i in perm]

        n_train = int(n * train_r)
        n_val = int(n * val_r)
        n_test = n - n_train - n_val

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:n_train + n_val + n_test]

        assignments[fold] = {
            "train": train_samples,
            "validate": val_samples,
            "test": test_samples,
        }

    return assignments


def write_split_samples(
    split_dir: Path,
    samples: List[SampleInfo],
    split_name: str,
) -> List[dict]:
    records = []
    for s in samples:
        image = read_png(s.image_path)
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2)

        if s.has_label and s.label_path is not None:
            label = read_png(s.label_path)
        else:
            label = np.full(image.shape[:2], -1, dtype=np.float32)

        if s.has_mask and s.mask_path is not None:
            mask = read_png(s.mask_path)
        else:
            mask = np.ones(image.shape[:2], dtype=np.float32)

        out_path = split_dir / f"{s.stem}.npz"
        save_npz_sample(
            sample_path=out_path,
            image=image,
            label=label,
            mask=mask,
            has_label=s.has_label,
            has_mask=s.has_mask,
            source_stem=s.stem,
        )

        records.append(
            {
                "split": split_name,
                "stem": s.stem,
                "file": out_path.name,
                "has_label": s.has_label,
                "has_mask": s.has_mask,
            }
        )

    return records


def write_set(
    set_id: int,
    split_assignment: Dict[str, List[SampleInfo]],
    holdout_samples: List[SampleInfo],
    output_root: Path,
) -> None:
    set_dir = output_root / f"set_{set_id}"
    train_dir = set_dir / "train"
    val_dir = set_dir / "validate"
    test_dir = set_dir / "test"
    holdout_dir = set_dir / "holdout"

    if set_dir.exists():
        for npz in set_dir.rglob("*.npz"):
            npz.unlink()

    records = []
    records.extend(write_split_samples(train_dir, split_assignment["train"], "train"))
    records.extend(write_split_samples(val_dir, split_assignment["validate"], "validate"))
    records.extend(write_split_samples(test_dir, split_assignment["test"], "test"))
    records.extend(write_split_samples(holdout_dir, holdout_samples, "holdout"))

    manifest_by_split = {
        "train": [r for r in records if r["split"] == "train"],
        "validate": [r for r in records if r["split"] == "validate"],
        "test": [r for r in records if r["split"] == "test"],
        "holdout": [r for r in records if r["split"] == "holdout"],
    }

    for split, split_records in manifest_by_split.items():
        manifest_path = set_dir / f"manifest_{split}.json"
        manifest_path.write_text(json.dumps(split_records, indent=2), encoding="utf-8")

    summary = {
        "set_id": set_id,
        "counts": {k: len(v) for k, v in manifest_by_split.items()},
    }
    (set_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

### 4. Create the datasets
Create all sets and write manifests.

In [7]:
matched_samples, holdout_samples = collect_samples(
    images_dir=IMAGES_DIR,
    labels_dir=LABELS_DIR,
    masks_dir=MASKS_DIR,
    exts=IMAGE_EXTS,
)

assignments = create_fold_assignments(
    matched_samples=matched_samples,
    k=K,
    seed=SEED,
    ratios=SPLIT_RATIOS,
)

for set_id in range(K):
    write_set(
        set_id=set_id,
        split_assignment=assignments[set_id],
        holdout_samples=holdout_samples,
        output_root=OUTPUT_ROOT,
    )

print(f"Created {K} sets under: {OUTPUT_ROOT}")

Matched count: 24
Holdout count: 5
Matched examples: ['121', '241', '270', '272', '274']
Holdout examples: ['535', '537', '539', '551', '553']
Created 4 sets under: data/sets


### 5. Add Quick Validation Checks
Validate folder structure, NPZ schema, and split counts.

In [8]:
required_splits = ["train", "validate", "test", "holdout"]
required_keys = {"image", "label", "mask", "has_label", "has_mask", "source_stem"}

for set_id in range(K):
    set_dir = OUTPUT_ROOT / f"set_{set_id}"
    assert set_dir.exists(), f"Missing set dir: {set_dir}"

    for split in required_splits:
        split_dir = set_dir / split
        assert split_dir.exists(), f"Missing split dir: {split_dir}"
        npz_files = sorted(split_dir.glob("*.npz"))
        assert len(npz_files) > 0 or split == "holdout", f"No files found in {split_dir}"

        if npz_files:
            sample = np.load(npz_files[0])
            assert required_keys.issubset(set(sample.files)), (
                f"Missing keys in {npz_files[0]}: {required_keys - set(sample.files)}"
            )

    summary_path = set_dir / "summary.json"
    assert summary_path.exists(), f"Missing summary: {summary_path}"

print("Validation checks passed.")

Validation checks passed.


Load one saved sample to confirm round-trip correctness and inspect metadata.

In [9]:
example_set = OUTPUT_ROOT / "set_0"
example_npz = next((example_set / "train").glob("*.npz"), None)

if example_npz is not None:
    sample = np.load(example_npz)
    print("Sample file:", example_npz)
    print("Keys:", sample.files)
    print("image shape:", sample["image"].shape, sample["image"].dtype)
    print("label shape:", sample["label"].shape, sample["label"].dtype)
    print("mask shape:", sample["mask"].shape, sample["mask"].dtype)
    print("has_label:", bool(sample["has_label"]))
    print("has_mask:", bool(sample["has_mask"]))
    print("source_stem:", str(sample["source_stem"]))
else:
    print("No train sample found yet. Run the execution cell first.")

Sample file: data/sets/set_0/train/303.npz
Keys: ['image', 'label', 'mask', 'has_label', 'has_mask', 'source_stem']
image shape: (256, 256, 3) uint8
label shape: (256, 256) uint8
mask shape: (256, 256) uint8
has_label: True
has_mask: True
source_stem: 303
